<a href="https://colab.research.google.com/github/luffzyzz/LFA-Agente-Especialista/blob/main/Especialista_em_solo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Execute primeiro a célula abaixo, depois a célula subsequente

In [ ]:
from datetime import datetime

# ---------- 1. Utilitários de classificação -------------------------
def classificar(valor, limites, nomes):
    """Devolve o nome da classe cujo limite superior (<=) contém o valor."""
    for limite, nome in zip(limites, nomes):
        if valor <= limite:
            return nome
    return nomes[-1]

NIVEIS = ["muito baixo", "baixo", "médio", "bom", "muito bom"]

def limites_fosforo(argila):
    """Limites superiores das 4 primeiras classes de P (Mehlich-1, mg/dm³) por teor de argila."""
    if argila > 60:
        return [2.7, 5.4, 8.0, 12.0]
    if argila > 35:
        return [4.0, 8.0, 12.0, 18.0]
    if argila > 15:
        return [6.6, 12.0, 20.0, 30.0]
    return [10.0, 20.0, 30.0, 45.0]

def classe_relevo(declividade):
    if declividade < 3:
        return "plano"
    if declividade < 8:
        return "suave ondulado"
    if declividade < 20:
        return "ondulado"
    if declividade < 45:
        return "forte ondulado"
    if declividade < 75:
        return "montanhoso"
    return "escarpado"

CAMPOS = ["ph", "mo", "p", "k", "ca", "mg", "al", "hal", "argila",
          "declividade", "v2", "prnt"]

# ---------- 2. Motor de inferência ----------------------------------
def interpretar(d, ca_sub=None, m_sub=None):
    """d: dicionário com os campos de CAMPOS.
    ca_sub / m_sub: Ca (cmolc/dm³) e saturação por Al (%) da camada de 20-40 cm (opcionais)."""
    for c in CAMPOS:
        if d.get(c) is None:
            raise ValueError(f"Campo ausente: {c}")
    if not (d["prnt"] > 0 and d["hal"] >= 0 and d["argila"] >= 0):
        raise ValueError("Verifique PRNT (> 0), H+Al (>= 0) e argila (>= 0).")

    crit, lim, rec, dec, avisos = [], [], [], [], []

    # --- fatos derivados ---
    k_cmolc = d["k"] / 391.0
    sb = d["ca"] + d["mg"] + k_cmolc
    t_efet = sb + d["al"]
    T = sb + d["hal"]
    if T <= 0 or t_efet <= 0:
        raise ValueError("CTC inválida: confira Ca, Mg, K, Al e H+Al.")
    V = 100 * sb / T
    m = 100 * d["al"] / t_efet

    # --- classes ---
    c_ph = ("muito baixo" if d["ph"] < 4.5 else "baixo" if d["ph"] <= 5.4 else
            "bom" if d["ph"] <= 6.0 else "alto" if d["ph"] <= 7.0 else "muito alto")
    c_mo = classificar(d["mo"], [0.7, 2.0, 4.0, 7.0], NIVEIS)
    c_p = classificar(d["p"], limites_fosforo(d["argila"]), NIVEIS)
    c_k = classificar(d["k"], [15, 40, 70, 120], NIVEIS)
    c_v = classificar(V, [20, 40, 60, 80], NIVEIS)
    c_T = classificar(T, [1.6, 4.3, 8.6, 15.0], NIVEIS)
    c_m = classificar(m, [15, 30, 50, 75],
                      ["muito baixa", "baixa", "média", "alta", "muito alta"])
    relevo = classe_relevo(d["declividade"])

    # --- verificações de coerência ---
    if V > 100:
        avisos.append(f"V = {V:.1f}% é maior que 100%: confira os dados do laudo.")
    if d["ph"] > 6.0 and m > 10:
        avisos.append("pH acima de 6,0 com saturação por alumínio acima de 10% é atípico: confira o laudo.")

    # --- R1: acidez ativa (pH em água) ---
    if d["ph"] < 4.5:
        crit.append("pH muito baixo")
        dec.append(("R1", "pH < 4,5", "fator CRÍTICO (acidez muito elevada)",
                    f"pH = {d['ph']:.1f}"))
    elif d["ph"] <= 5.4:
        lim.append("pH baixo")
        dec.append(("R1", "4,5 <= pH <= 5,4", "fator LIMITANTE (acidez elevada)",
                    f"pH = {d['ph']:.1f}"))
    elif d["ph"] > 7.0:
        lim.append("pH alto")
        dec.append(("R1", "pH > 7,0", "fator LIMITANTE (pH muito alto)", f"pH = {d['ph']:.1f}"))
        rec.append("pH alto: investigar a causa (por exemplo, excesso de corretivo ou sais) antes de qualquer nova calagem.")

    # --- R2: saturação por alumínio ---
    if m > 50:
        crit.append(f"saturação por alumínio alta (m = {m:.0f}%)")
        dec.append(("R2", "m > 50%", "fator CRÍTICO (alumínio tóxico)", f"m = {m:.1f}%"))
    elif m > 30:
        lim.append(f"saturação por alumínio média (m = {m:.0f}%)")
        dec.append(("R2", "30% < m <= 50%", "fator LIMITANTE", f"m = {m:.1f}%"))

    # --- R3: saturação por bases ---
    if V <= 20:
        crit.append(f"saturação por bases muito baixa (V = {V:.0f}%)")
        dec.append(("R3", "V <= 20%", "fator CRÍTICO", f"V = {V:.1f}%"))
    elif V <= 40:
        lim.append(f"saturação por bases baixa (V = {V:.0f}%)")
        dec.append(("R3", "20% < V <= 40%", "fator LIMITANTE", f"V = {V:.1f}%"))

    # --- R4: fósforo (depende da argila) ---
    lp = limites_fosforo(d["argila"])
    if c_p == "muito baixo":
        crit.append("fósforo muito baixo")
        dec.append(("R4", f"P <= {lp[0]} mg/dm³ (argila {d['argila']:.0f}%)", "fator CRÍTICO", f"P = {d['p']} mg/dm³"))
    elif c_p == "baixo":
        lim.append("fósforo baixo")
        dec.append(("R4", f"{lp[0]} < P <= {lp[1]} mg/dm³ (argila {d['argila']:.0f}%)", "fator LIMITANTE", f"P = {d['p']} mg/dm³"))
    if c_p in ("muito baixo", "baixo"):
        rec.append("Adubação fosfatada de correção. A dose depende da cultura, da produtividade esperada e do manual de recomendação do seu estado.")
        dec.append(("R4a", "P nas classes muito baixo ou baixo", "recomendar adubação fosfatada de correção", "a dose não é calculada pelo sistema"))

    # --- R5: potássio ---
    if c_k == "muito baixo":
        crit.append("potássio muito baixo")
        dec.append(("R5", "K <= 15 mg/dm³", "fator CRÍTICO", f"K = {d['k']} mg/dm³"))
    elif c_k == "baixo":
        lim.append("potássio baixo")
        dec.append(("R5", "15 < K <= 40 mg/dm³", "fator LIMITANTE", f"K = {d['k']} mg/dm³"))
    if c_k in ("muito baixo", "baixo"):
        txt = "Adubação potássica conforme a cultura e o manual de recomendação do estado."
        if T < 4.0:
            txt += " CTC baixa (< 4,0 cmolc/dm³): há maior risco de perdas de potássio; o técnico deve avaliar o parcelamento."
            dec.append(("R5a", "K baixo ou muito baixo E T < 4,0 cmolc/dm³", "alertar para risco de perdas e avaliar parcelamento", f"T = {T:.2f}"))
        rec.append(txt)

    # --- R6: matéria orgânica ---
    if c_mo == "muito baixo":
        crit.append("matéria orgânica muito baixa")
        dec.append(("R6", "MO <= 0,7 dag/kg", "fator CRÍTICO", f"MO = {d['mo']} dag/kg"))
    elif c_mo == "baixo":
        lim.append("matéria orgânica baixa")
        dec.append(("R6", "0,7 < MO <= 2,0 dag/kg", "fator LIMITANTE", f"MO = {d['mo']} dag/kg"))
    if c_mo in ("muito baixo", "baixo"):
        rec.append("Matéria orgânica baixa: incluir no plano de manejo o aumento do aporte de resíduos vegetais (palhada, rotação, adubação verde).")

    # --- R7: veredito químico ---
    if crit or len(lim) >= 2:
        veredito = "SOLO INADEQUADO para cultivo sem correção"
        nivel = "inadequado"
        dec.append(("R7", "há 1 ou mais fatores críticos OU 2 ou mais fatores limitantes",
                    "solo INADEQUADO", f"críticos: {len(crit)}; limitantes: {len(lim)}"))
    elif len(lim) == 1:
        veredito = "SOLO ADEQUADO COM RESTRIÇÃO"
        nivel = "restricao"
        dec.append(("R7", "exatamente 1 fator limitante e nenhum crítico",
                    "solo ADEQUADO COM RESTRIÇÃO", f"limitante: {lim[0]}"))
    else:
        veredito = "SOLO ADEQUADO (pela química do laudo)"
        nivel = "adequado"
        dec.append(("R7", "nenhum fator crítico ou limitante", "solo ADEQUADO", "todos os atributos avaliados fora das classes críticas"))

    # --- R8: calagem pelo método da saturação por bases ---
    nc = (d["v2"] - V) * T / 100.0 * (100.0 / d["prnt"])
    indicador_acidez = d["ph"] <= 5.4 or m > 30 or V <= 40
    if indicador_acidez and nc > 0:
        rec.insert(0, f"Calagem: cerca de {nc:.1f} t/ha de calcário com PRNT de {d['prnt']:.0f}% "
                      f"(elevar V de {V:.0f}% para {d['v2']:.0f}%). O tipo de calcário (calcítico, magnesiano ou dolomítico) "
                      "deve considerar o teor de magnésio do laudo; aplicar antes do plantio.")
        dec.append(("R8", "(pH <= 5,4 OU m > 30% OU V <= 40%) E V < V2",
                    "recomendar calagem com NC = (V2 − V) × T / 100 × 100 / PRNT",
                    f"V = {V:.1f}%, V2 = {d['v2']:.0f}%, T = {T:.2f}, PRNT = {d['prnt']:.0f}% → NC = {nc:.2f} t/ha"))

    # --- R9: gessagem (exige análise da camada de 20-40 cm) ---
    if ca_sub is not None and m_sub is not None:
        if ca_sub <= 0.4 or m_sub > 20:
            rec.append("Gessagem: a camada de 20 a 40 cm apresenta cálcio baixo e/ou saturação por alumínio elevada; "
                       "avaliar o uso de gesso agrícola com a dose definida por um técnico.")
            dec.append(("R9", "Ca (20-40 cm) <= 0,4 cmolc/dm³ OU m (20-40 cm) > 20%",
                        "recomendar avaliação de gessagem",
                        f"Ca subsuperfície = {ca_sub}; m subsuperfície = {m_sub}%"))
        else:
            dec.append(("R9", "Ca (20-40 cm) <= 0,4 cmolc/dm³ OU m (20-40 cm) > 20%",
                        "gessagem NÃO indicada", f"Ca subsuperfície = {ca_sub}; m subsuperfície = {m_sub}%"))
    elif m > 30:
        rec.append("Saturação por alumínio elevada na camada superficial: coletar também amostra de 20 a 40 cm para avaliar a necessidade de gessagem.")
        dec.append(("R9", "m (0-20 cm) > 30% E sem análise de 20-40 cm", "pedir amostra da subsuperfície", f"m = {m:.1f}%"))

    # --- R10: relevo ---
    dc = d["declividade"]
    if dc >= 25:
        rec.append(f"Relevo {relevo} ({dc:.0f}%): declividade muito alta para culturas anuais. Buscar orientação técnica sobre o uso da área e sobre a legislação ambiental aplicável.")
        dec.append(("R10", "declividade >= 25%", "uso agrícola intensivo não indicado; buscar orientação técnica", f"declividade = {dc}%"))
    elif dc >= 12:
        rec.append(f"Relevo {relevo} ({dc:.0f}%): risco de erosão elevado. Adotar práticas conservacionistas mais intensivas (como terraceamento e plantio em nível) com projeto de um técnico"
                   + (", junto com a correção do solo, para que calcário e adubo não sejam perdidos por erosão." if nivel == "inadequado" else "."))
        dec.append(("R10", "12% <= declividade < 25%", "práticas conservacionistas intensivas", f"declividade = {dc}%"))
    elif dc >= 6:
        rec.append(f"Relevo {relevo} ({dc:.0f}%): adotar práticas conservacionistas simples (plantio em nível, cobertura do solo e plantio direto).")
        dec.append(("R10", "6% <= declividade < 12%", "práticas conservacionistas simples", f"declividade = {dc}%"))
    else:
        dec.append(("R10", "declividade < 6%", "sem exigência de prática conservacionista específica além do manejo habitual", f"declividade = {dc}%"))
    if nivel == "adequado" and dc >= 6:
        rec.append("A química do solo é adequada, mas o relevo exige monitoramento de erosão.")
        dec.append(("R11", "solo adequado E declividade >= 6%", "monitorar erosão", f"declividade = {dc}%"))

    if not rec:
        rec.append("Manter o manejo atual e repetir a análise de solo no ciclo habitual.")
        dec.append(("R12", "nenhuma ação corretiva disparada", "manter manejo e reanalisar", "-"))

    tabela = [
        ("pH em água", f"{d['ph']:.1f}", c_ph),
        ("Matéria orgânica (dag/kg)", f"{d['mo']}", c_mo),
        ("Fósforo, Mehlich-1 (mg/dm³)", f"{d['p']}", c_p),
        ("Potássio, Mehlich-1 (mg/dm³)", f"{d['k']}", c_k),
        ("CTC a pH 7, T (cmolc/dm³)", f"{T:.2f}", c_T),
        ("Saturação por bases, V (%)", f"{V:.1f}", c_v),
        ("Saturação por alumínio, m (%)", f"{m:.1f}", c_m),
    ]
    return dict(veredito=veredito, nivel=nivel, criticos=crit, limitantes=lim,
                recomendacoes=rec, decisoes=dec, avisos=avisos, tabela=tabela,
                derivados=dict(SB=sb, t=t_efet, T=T, V=V, m=m, K_cmolc=k_cmolc),
                relevo=relevo, entrada=dict(d), ca_sub=ca_sub, m_sub=m_sub)

# ---------- 3. Relatório de decisão (arquivo de texto) --------------
def gerar_relatorio(r, caminho=None):
    """Escreve um .txt explicando passo a passo como o sistema decidiu."""
    agora = datetime.now()
    if caminho is None:
        caminho = "relatorio_decisao_solo_" + agora.strftime("%Y%m%d_%H%M%S") + ".txt"
    d, v = r["entrada"], r["derivados"]
    L = []
    L.append("RELATÓRIO DE DECISÃO DO SISTEMA ESPECIALISTA — INTERPRETAÇÃO DE LAUDO DE SOLO")
    L.append("Gerado em " + agora.strftime("%d/%m/%Y %H:%M"))
    L.append("=" * 78)
    L.append("")
    L.append("1. DADOS INFORMADOS")
    nomes = {"ph": "pH em água", "mo": "Matéria orgânica (dag/kg)", "p": "Fósforo Mehlich-1 (mg/dm³)",
             "k": "Potássio Mehlich-1 (mg/dm³)", "ca": "Cálcio (cmolc/dm³)", "mg": "Magnésio (cmolc/dm³)",
             "al": "Alumínio (cmolc/dm³)", "hal": "H+Al (cmolc/dm³)", "argila": "Argila (%)",
             "declividade": "Declividade (%)", "v2": "V desejada, V2 (%)", "prnt": "PRNT do calcário (%)"}
    for c in CAMPOS:
        L.append(f"   - {nomes[c]}: {d[c]}")
    if r["ca_sub"] is not None:
        L.append(f"   - Cálcio na camada 20-40 cm (cmolc/dm³): {r['ca_sub']}")
        L.append(f"   - Saturação por Al na camada 20-40 cm (%): {r['m_sub']}")
    L.append("")
    L.append("2. VALORES CALCULADOS PELO SISTEMA")
    L.append(f"   - K em cmolc/dm³ = K (mg/dm³) / 391 = {v['K_cmolc']:.3f}")
    L.append(f"   - Soma de bases SB = Ca + Mg + K = {v['SB']:.2f} cmolc/dm³")
    L.append(f"   - CTC efetiva t = SB + Al = {v['t']:.2f} cmolc/dm³")
    L.append(f"   - CTC a pH 7 T = SB + (H+Al) = {v['T']:.2f} cmolc/dm³")
    L.append(f"   - Saturação por bases V = 100 × SB / T = {v['V']:.1f} %")
    L.append(f"   - Saturação por alumínio m = 100 × Al / t = {v['m']:.1f} %")
    L.append(f"   - Classe de relevo pela declividade: {r['relevo']}")
    L.append("")
    L.append("3. CLASSIFICAÇÃO DE CADA ATRIBUTO")
    for nome, valor, classe in r["tabela"]:
        L.append(f"   - {nome}: {valor} → {classe}")
    L.append("")
    L.append("4. REGRAS ACIONADAS (raciocínio do sistema, na ordem em que foram avaliadas)")
    for reg, se, entao, motivo in r["decisoes"]:
        L.append(f"   [{reg}] SE {se}")
        L.append(f"        ENTÃO {entao}")
        L.append(f"        Porque: {motivo}")
    L.append("")
    L.append("5. CONCLUSÃO")
    L.append(f"   Veredito: {r['veredito']}")
    L.append(f"   Fatores críticos: {', '.join(r['criticos']) if r['criticos'] else 'nenhum'}")
    L.append(f"   Fatores limitantes: {', '.join(r['limitantes']) if r['limitantes'] else 'nenhum'}")
    L.append("   Manejo indicado:")
    for i, x in enumerate(r["recomendacoes"], 1):
        L.append(f"     {i}. {x}")
    if r["avisos"]:
        L.append("   Avisos sobre os dados:")
        for a in r["avisos"]:
            L.append(f"     - {a}")
    L.append("")
    L.append("6. OBSERVAÇÕES")
    L.append("   - O sistema não calcula doses de adubo; elas dependem da cultura, da produtividade esperada e do manual do seu estado.")
    L.append("   - O critério que transforma as classes em veredito (um fator crítico, ou dois ou mais limitantes) é uma regra do próprio sistema.")
    L.append("   - Este relatório apoia, e não substitui, o parecer de um engenheiro agrônomo ou técnico habilitado.")
    with open(caminho, "w", encoding="utf-8") as f:
        f.write("\n".join(L) + "\n")
    return caminho

In [ ]:
import ipywidgets as w
from IPython.display import display, HTML

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

_estilo = {"description_width": "260px"}
_layout = w.Layout(width="480px")

def campo(rotulo, valor, passo=0.1):
    return w.FloatText(value=valor, description=rotulo, step=passo, style=_estilo, layout=_layout)

f = {
    "ph":          campo("pH em água", 5.0),
    "mo":          campo("Matéria orgânica (dag/kg)", 1.8),
    "p":           campo("Fósforo, Mehlich-1 (mg/dm³)", 4.0),
    "k":           campo("Potássio, Mehlich-1 (mg/dm³)", 47.0, 1),
    "ca":          campo("Cálcio (cmolc/dm³)", 1.2),
    "mg":          campo("Magnésio (cmolc/dm³)", 0.4),
    "al":          campo("Alumínio, Al³⁺ (cmolc/dm³)", 1.0),
    "hal":         campo("Acidez potencial, H+Al (cmolc/dm³)", 7.5),
    "argila":      campo("Argila (%)", 40.0, 1),
    "declividade": campo("Declividade do terreno (%)", 12.0, 1),
    "v2":          campo("V desejada, V2 (%) — depende da cultura", 60.0, 1),
    "prnt":        campo("PRNT do calcário (%)", 80.0, 1),
}
usar_sub = w.Checkbox(value=False, description="Tenho análise da camada de 20 a 40 cm (para avaliar gessagem)", indent=False,
                      layout=w.Layout(width="620px"))
ca_sub = campo("Cálcio 20-40 cm (cmolc/dm³)", 0.3)
m_sub = campo("Saturação por Al 20-40 cm (%)", 30.0, 1)
caixa_sub = w.VBox([ca_sub, m_sub], layout=w.Layout(display="none"))
usar_sub.observe(lambda c: setattr(caixa_sub.layout, "display", "block" if c["new"] else "none"), names="value")

botao = w.Button(description="Interpretar laudo", button_style="success", icon="check",
                 layout=w.Layout(width="220px"))
saida = w.Output()

def _html(txt, cor, fundo):
    return HTML(f'<div style="padding:12px;border-radius:8px;background:{fundo};color:{cor};'
                f'font-size:15px;margin:6px 0">{txt}</div>')

def ao_clicar(_):
    saida.clear_output()
    with saida:
        try:
            dados = {k: float(v.value) for k, v in f.items()}
            r = interpretar(dados,
                            ca_sub=float(ca_sub.value) if usar_sub.value else None,
                            m_sub=float(m_sub.value) if usar_sub.value else None)
        except ValueError as e:
            display(_html("⚠ " + str(e), "#8a1c14", "#fde4e2"))
            return
        cores = {"inadequado": ("#8a1c14", "#fde4e2"), "restricao": ("#7a4a00", "#fff1d6"),
                 "adequado": ("#17653a", "#e1f3e6")}
        c, fundo = cores[r["nivel"]]
        display(_html(f"<b style='font-size:18px'>{r['veredito']}</b>", c, fundo))
        for a in r["avisos"]:
            display(_html("⚠ " + a, "#7a4a00", "#fff1d6"))
        itens = "".join(f"<li style='margin:6px 0'>{x}</li>" for x in r["recomendacoes"])
        display(HTML(f"<h4 style='margin:12px 0 4px'>Manejo recomendado</h4><ol>{itens}</ol>"))
        caminho = gerar_relatorio(r)
        display(HTML(f"<p>📄 A explicação de como o sistema decidiu foi salva no arquivo "
                     f"<b>{caminho}</b> (pasta de arquivos do Colab).</p>"))
        if colab_files is not None:
            try:
                colab_files.download(caminho)
            except Exception:
                pass

botao.on_click(ao_clicar)
display(HTML("<h3>Sistema especialista: interpretação de laudos de solo</h3>"
             "<p>Informe os dados do laudo (extrator Mehlich-1) e do terreno.</p>"))
display(w.VBox(list(f.values()) + [usar_sub, caixa_sub, botao, saida]))